
```python
'''Recording trajectories and storing them into a databaseself.

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
ros2 launch object_localization box_localization_launch.py

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
python /home/imitlearn/lfd_ws/src/nocode_robot_programming/main.py

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
cd src
code .
'''
```

WARNING!!! When restarting the Jupyter kernel, you might need to kill it's residual proceess with: `pkill -9 -f ipykernel_launcher`

In [ ]:
#!/usr/bin/env python3
import rclpy
rclpy.init()
from skills_manager.risk_aware_lfd.ralfd import RALfD
from skills_manager.ros_param_manager import set_remote_parameters
from nocode_robot_programming.state_decision import StateDecision

lfd = RALfD(state_decider=StateDecision())
lfd.start()
lfd.keyboard_start()
lfd.joy_start()
lfd.teleop_start()

In [ ]:
set_remote_parameters(lfd, ["position_x", "position_y", "position_z", "orientation_x", "orientation_y", "orientation_z", "orientation_w"],
[0.4, 0.0, 0.4, 1.0, 0.0, 0.0, 0.0], server="localizer_node")
lfd.move_template_start()

In [ ]:
name_skill = lfd.declare_parameter_and_get('name_skill', "peg_pick")
print(f"Recording skill: {name_skill}", flush=True)

In [ ]:
lfd.skill_exists(name_skill)

In [ ]:
if not lfd.skill_exists(name_skill):
    lfd.traj_rec()
    lfd.save(name_skill)
    input("ENTER to replay")

In [ ]:
name_skill

In [ ]:
lfd.move_template_start()
success = lfd.play_skill(name_skill, None, localize_box=False)

In [ ]:
rclpy.shutdown()

In [ ]:
from video_embedding.utils import set_session, get_all_names
from video_embedding.models.video_embedder import VideoEmbedder
from video_embedding.models.video_embedding_dataset import load_dataloader

In [ ]:
video_embedder = VideoEmbedder(
    name=name_skill,
    latent_dim=12,
    learning_rate=0.001,
    nn_model="Autoencoder3",
)
train_videos = get_all_names(name_skill)
dataloader = load_dataloader(train_videos, batch_size=64)
video_embedder.train(dataloader, train_videos, num_epochs=50)
video_embedder.save_model()

In [ ]:
from risk_estimation.result_evaluator import ResultEvaluator
from risk_estimation.datasets.risk_feature_extractor import *
from risk_estimation.datasets.risk_dataloader import RiskEstimationDataset as D
from risk_estimation.datasets.frame_dropping import *
from risk_estimation.models.mlp_risk_estimator import MLPRiskEstimator, MLPRiskEstimator2
from risk_estimation.models.gp_risk_estimator import GPRiskEstimator, TwinGPRiskEstimator
from risk_estimation.models.dist_risk_estimator import *
from risk_estimation.models.resnet_risk_estimator import ResNetRiskEstimator
from risk_estimation.models.safety_layer import get_risk_estimator
from video_embedding.utils import all_trial_names, all_test_names, visualize_labelled_video, visualize_labelled_video_frame, get_session
from video_embedding.models.video_embedder import VideoEmbedder

from risk_estimation.models.risk_estimator import *
from video_embedding.models.nerual_networks.autoencoder import *

dataloader = D.load_dataloader(all_trial_names(name_skill) + all_test_names(name_skill), 
    video_embedder, 40, OnlyLabelledFramesDroppingPolicy, StampedLatentObservationsRiskLabels)

risk_estimator = get_risk_estimator(
    approach="GP",
    skill_name=video_embedder.name,
    xdim=StampedLatentObservationsRiskLabels.xdim(12),
    video_embedder=video_embedder,
    out_assessment="cautious",
    train_epoch = 400,
)
risk_estimator.set_dataloaders_for_validation([
        dataloader,
    ], names=["test"])
risk_estimator.training_loop(dataloader)

In [ ]:
dataloader.dataset.X

In [ ]:
dataloader.dataset.X.shape

In [ ]:
lfd.save(name_skill+"_new")

In [ ]:
lfd.desk.lock()

In [ ]:
lfd.desk.unlock()